##Synthetic Dataset Generator##

features_config **List**
Each element in features_config is a dictionary describing how to generate a single column. You can have as many columns (features) as you like.

**Distribution Parameters**

You specify the distribution in the "distribution" key (e.g., "normal", "uniform", "exponential").
Distribution-specific parameters (like loc, scale, low, high, etc.) go in "dist_params".

**Data Type**

By default, "data_type": "float" simply creates floating-point values.
You can specify "int" (the script will round values before converting to integers) or "string" (converts numbers to strings).
Extend _convert_to_type if you need booleans, dates, categoricals, etc.

**Noise**

"noise_std" injects Gaussian noise with mean=0 and standard deviation = noise_std.
If you set "noise_std": 0.0, no extra noise is added.

**Missing Values**

"missing_rate" is the fraction of values (0 to 1) replaced with NaN.
e.g., missing_rate=0.1 will make roughly 10% of the entries in that column NaN.

**Random Seeds**

You can provide a global random_seed to make the entire dataset reproducible.
You can also override the seed for an individual feature by putting "random_seed": 101 inside its config.

**Extending the Code**

* More Distributions: Add more options to _generate_base_distribution.
* More Data Types: Extend _convert_to_type for specialized data (dates, times, categoricals).
* Complex Features: You can also generate multi-modal distributions or custom transformations if you need more realism.
This structure offers flexibility and clarity, letting you generate large, multi-feature datasets with varied distributions, data types, noise, and missing values—all from a single function call.

In [1]:
import numpy as np
import pandas as pd

def generate_synthetic_dataset(
    num_samples=1000,
    features_config=None,
    random_seed=None
):
    """
    Generates a synthetic dataset with multiple columns (features),
    each with its own distribution, data type, noise, and missing rate.

    Parameters
    ----------
    num_samples : int, optional
        Number of data points (rows) to generate.
    features_config : list of dict, optional
        A list of feature configuration dictionaries.
        Each dictionary can have the following keys:
          - name (str): The column name.
          - distribution (str): Distribution type (e.g. 'normal', 'uniform', 'exponential', etc.).
          - dist_params (dict): Dictionary of parameters for the chosen distribution.
                               Defaults to None or empty dict if not provided.
          - data_type (str): 'int', 'float', or 'string' (basic type conversion).
                             For more sophisticated data types, implement custom logic.
          - noise_std (float): Standard deviation of Gaussian noise to add (defaults to 0.0 for no noise).
          - missing_rate (float): Fraction of data in the column to randomly replace with NaN.
          - random_seed (int): Optionally override the main random seed for this column specifically.
    random_seed : int, optional
        Seed for the random number generator, to make results reproducible.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing all requested features (columns).
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    if features_config is None or not isinstance(features_config, list):
        raise ValueError("features_config must be a list of feature configurations.")

    df = pd.DataFrame()

    for feature in features_config:
        col_name = feature.get('name', 'feature')
        distribution = feature.get('distribution', 'normal')
        dist_params = feature.get('dist_params', {})
        data_type = feature.get('data_type', 'float')
        noise_std = feature.get('noise_std', 0.0)
        missing_rate = feature.get('missing_rate', 0.0)

        # If the user provided a separate random seed for this column, override
        feature_seed = feature.get('random_seed', None)
        if feature_seed is not None:
            np.random.seed(feature_seed)

        # Generate base data
        col_data = _generate_base_distribution(
            distribution=distribution,
            dist_params=dist_params,
            num_samples=num_samples
        )

        # Revert seed if needed
        if feature_seed is not None and random_seed is not None:
            np.random.seed(random_seed)

        # Add noise
        if noise_std > 0.0:
            noise = np.random.normal(loc=0.0, scale=noise_std, size=num_samples)
            col_data += noise

        # Convert to requested data type
        col_data = _convert_to_type(col_data, data_type)

        col_data = np.round(col_data).astype(float)

        # Generate missing values
        if missing_rate > 0.0:
            num_missing = int(np.floor(missing_rate * num_samples))
            missing_indices = np.random.choice(num_samples, size=num_missing, replace=False)
            col_data[missing_indices] = np.nan

        # Create a Series and add to DataFrame
        df[col_name] = col_data

    return df

def _generate_base_distribution(distribution, dist_params, num_samples):
    """
    Generates base distribution data as a NumPy array.
    Supported distributions can be expanded as needed.
    """
    distribution = distribution.lower()
    if distribution == 'normal':
        # default loc=0, scale=1 if not specified
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.normal(loc=loc, scale=scale, size=num_samples)
    elif distribution == 'uniform':
        # default low=0, high=1 if not specified
        low = dist_params.get('low', 0.0)
        high = dist_params.get('high', 1.0)
        return np.random.uniform(low=low, high=high, size=num_samples)
    elif distribution == 'exponential':
        # default scale=1
        scale = dist_params.get('scale', 1.0)
        return np.random.exponential(scale=scale, size=num_samples)
    elif distribution == 'logistic':
        # default loc=0, scale=1
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.logistic(loc=loc, scale=scale, size=num_samples)
    elif distribution == 'beta':
        # default a=2, b=5
        a = dist_params.get('a', 2.0)
        b = dist_params.get('b', 5.0)
        return np.random.beta(a, b, size=num_samples)
    # Add more distributions here if needed
    else:
        raise ValueError(f"Unsupported distribution: {distribution}")


def _convert_to_type(data_array, data_type):
    """
    Converts a NumPy array to the specified data type.
    Basic version supporting int, float, and string.
    Handles NaN for integer types by converting to float64.
    """
    data_type = data_type.lower()
    if data_type == 'float':
        return data_array.astype(float)
    elif data_type == 'int':
        # If there are NaNs, convert to float64 to accommodate them
        if np.isnan(data_array).any():
            return data_array.astype(np.float64)  # Change to float64 to hold NaN
        else:
            return np.round(data_array).astype(int)
    elif data_type == 'string':
        return data_array.astype(str)
    else:
        raise ValueError(f"Unsupported data_type: {data_type}")


if __name__ == "__main__":
    # Example usage: generate 5,000 rows with 3 different features
    features_config = [
        {
            'name': 'height_cm',
            'distribution': 'normal',
            'dist_params': {'loc': 170, 'scale': 10},
            'data_type': 'float',
            'noise_std': 2.0,
            'missing_rate': 0.05
        },
        {
            'name': 'age',
            'distribution': 'normal',
            'dist_params': {'loc': 40, 'scale': 12},
            'data_type': 'int',
            'noise_std': 5.0,
            'missing_rate': 0.1
        },
        {
            'name': 'city_code',
            'distribution': 'uniform',
            'dist_params': {'low': 1, 'high': 10},
            'data_type': 'int',
            'missing_rate': 0.0
        }
    ]

    df_synthetic = generate_synthetic_dataset(
        num_samples=5000,
        features_config=features_config,
        random_seed=42
    )

    print(df_synthetic.head(15))
    print(df_synthetic.describe(include='all'))


    height_cm   age  city_code
0       174.0  15.0        7.0
1       168.0  55.0        6.0
2       173.0  51.0        6.0
3       185.0  39.0        1.0
4       169.0  62.0       10.0
5       165.0  24.0        9.0
6       188.0  56.0        7.0
7       179.0  23.0        2.0
8       164.0  56.0        3.0
9       175.0  18.0        3.0
10      169.0  59.0        6.0
11      167.0  58.0        3.0
12      172.0  21.0        7.0
13      150.0  59.0        6.0
14        NaN  53.0        1.0
        height_cm          age    city_code
count  4750.00000  4500.000000  5000.000000
mean    170.01200    40.074000     5.476600
std      10.14986    12.709383     2.615267
min     135.00000    -3.000000     1.000000
25%     163.00000    32.000000     3.000000
50%     170.00000    40.000000     6.000000
75%     177.00000    49.000000     8.000000
max     209.00000    84.000000    10.000000
